# BOQ Intelligence Platform — MVP Workflow

Notebook นี้ใช้ MySQL และ SQL สำหรับการสรุปผล โดยเรียก logic ชุดเดียวกับ `main.py` ไม่มี Streamlit

ลำดับ: ตรวจฐานข้อมูล → โหลด Dataset 2/ราคาหน้าร้าน → เลือกบ้าน/ห้อง/รายชิ้น → รับ BOQ → ตรวจการจับคู่ → สาธิต CRUD → แสดง Dashboard, Insight และ Action Plan → ส่งออกผล

ใช้เฉพาะข้อมูลจริงจาก BOQ, API หรือ snapshot ราคาที่เก็บจากแหล่งที่ได้รับอนุญาต ไม่มีข้อมูลตัวอย่างสังเคราะห์ ตัวเลขและตาราง output จะถูกสร้างจากฐานข้อมูลเมื่อรันเซลล์

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from boq_platform.db import connection, query
from boq_platform.ingest import import_moc_prices, import_retail_csv, read_boq_excel
from boq_platform.projects import (create_project, add_room, add_boq_item, import_boq_rows,
    list_boq_items, confirm_boq_match, update_boq_item, delete_boq_item)
from boq_platform.matching import find_candidates
from boq_platform.analytics import generate_insights
from boq_platform.audit import append_event
from boq_platform.quality_review import list_price_review_queue, resolve_price_review
from boq_platform.config import EXPORT_DIR, LOG_DIR
from IPython.display import display, HTML
import pandas as pd
import matplotlib.pyplot as plt
print('Workspace:', ROOT)
with connection() as conn:
    cur = conn.cursor()
    cur.execute('SELECT VERSION()')
    print('MySQL:', cur.fetchone()[0])

## 1. เตรียมข้อมูลราคา

Dataset 2 เป็นราคาทางการเฉลี่ยรายจังหวัดรายเดือน ใช้รหัสจังหวัดและปี ค.ศ. ส่วนราคาจากร้านค้าให้กรอกไฟล์ `templates/retail_prices_template.csv` จากราคาที่ตรวจสอบได้จริง เก็บวัน สาขา หน่วย VAT ค่าขนส่ง และ URL ไว้ด้วย Notebook ไม่ scrape เว็บอัตโนมัติ

In [ ]:
load_moc = input('นำเข้าราคาทางการ Dataset 2 ตอนนี้หรือไม่? พิมพ์ Y/N: ').strip().lower() == 'y'
if load_moc:
    province = input('รหัสจังหวัด (เช่น 10): ').strip()
    year_from = int(input('ปีเริ่มต้น ค.ศ.: ').strip())
    year_to = int(input('ปีสิ้นสุด ค.ศ.: ').strip())
    moc_result = import_moc_prices(province, year_from, year_to)
    print(moc_result)
else:
    print('ข้ามการดึง API — ใช้ข้อมูลทางการที่มีอยู่ใน MySQL')

retail_csv = input('พาธ CSV snapshot ราคาจากร้านค้า (Enter เพื่อข้าม): ').strip().strip('\"')
if retail_csv:
    retail_result = import_retail_csv(retail_csv)
    print(retail_result)
else:
    print('ข้ามการนำเข้าราคาหน้าร้าน')

## 2. Data Cleansing ที่เกิดขึ้นตอนนำเข้า

1. เก็บ response/ไฟล์ดิบไว้ใน `data_raw/` และบันทึกชื่อไฟล์
2. ปรับรูปแบบชื่อวัสดุ/รหัสและเก็บชื่อดิบไว้ใน `raw_fields`
3. ปรับหน่วยด้วย mapping ที่กำหนด ไม่แปลงหน่วยแพ็กเกจที่ไม่มีข้อมูลขนาดบรรจุ
4. แปลงราคาเป็น decimal และแยก VAT/ราคาโปรโมชัน/ค่าขนส่ง
5. ปรับปีและวันให้เป็นมาตรฐาน โดยแปลง พ.ศ. เป็น ค.ศ. เมื่อจำเป็น
6. ตรวจข้อมูลซ้ำด้วย hash ของแหล่ง รหัสสินค้า วัน พื้นที่ หน่วย และราคา เพื่อให้การนำเข้าซ้ำไม่เพิ่มแถวซ้ำ
7. ปฏิเสธแถวที่ขาดชื่อ หน่วย วันที่ หรือราคาที่ถูกต้อง และติดธงค่าผิดปกติเพื่อตรวจ ไม่ลบ outlier อัตโนมัติ
8. เก็บสถานะคุณภาพและเหตุผล พร้อม append เหตุการณ์ลง `logs/audit.jsonl`

ตรวจจำนวนรายการที่นำเข้าและคุณภาพข้อมูลได้จากผลลัพธ์เซลล์ก่อนหน้าและ audit log

In [ ]:
review_rows = list_price_review_queue()
display(pd.DataFrame(review_rows))
if review_rows and input('ตรวจรายการราคา review หรือไม่? Y/N: ').strip().lower() == 'y':
    observation_id = int(input('Observation ID: ').strip())
    decision = input('พิมพ์ valid หรือ rejected: ').strip().lower()
    print(resolve_price_review(observation_id, decision, actor='notebook_user'))

## 3. เลือกขนาดงานและสร้าง Project

เลือก `house` สำหรับ BOQ ทั้งหลัง, `room` สำหรับห้อง/พื้นที่ และ `item` สำหรับเช็กรายชิ้น รหัส Project และ Item สร้างโดยระบบ ผู้ใช้ไม่ต้องกรอก ID เอง

In [ ]:
scope_label = input('ขอบเขต house / room / item: ').strip().lower()
scope_type = {'1':'house','2':'room','3':'item','บ้านทั้งหลัง':'house','ห้อง':'room','รายชิ้น':'item'}.get(scope_label, scope_label)
if scope_type not in {'house','room','item'}: raise ValueError('เลือก house, room หรือ item')
project_name = input('ชื่อโครงการ: ').strip() or 'BOQ Project'
province_code = input('รหัสจังหวัดโครงการ (Enter ถ้าไม่ระบุ): ').strip() or None
province_name = input('ชื่อจังหวัด (Enter ถ้าไม่ระบุ): ').strip() or None
budget = input('งบประมาณรวม (Enter ถ้าไม่ระบุ): ').strip() or None
project_id = create_project(project_name, scope_type, province_code, province_name, budget)
print('Project ID:', project_id)

## 4. รับรายการ BOQ

บ้านทั้งหลังหรือห้องที่มี Excel ให้อัปโหลดไฟล์ หากไม่มี Excel หรือเลือกขอบเขตรายชิ้น ให้กรอกชื่อ สเปก จำนวน หน่วย และราคา BOQ (ถ้ามี) ทีละรายการ พิมพ์ `DONE` เพื่อจบการกรอก ระบบจับคู่ชื่อ/รหัส/หน่วย และบันทึกหลักฐานทุก candidate

In [ ]:
boq_path = input('พาธไฟล์ BOQ .xlsx/.csv (Enter เพื่อกรอกเอง): ').strip().strip('\"')
if boq_path:
    boq_rows = read_boq_excel(boq_path)
    import_result = import_boq_rows(project_id, boq_rows)
    print(import_result)
else:
    room_id = None
    room_name = input('ชื่อห้อง/พื้นที่ (Enter ถ้าไม่แบ่งห้อง): ').strip()
    if room_name: room_id = add_room(project_id, room_name)
    manual_count = 0
    while True:
        name = input('ชื่อวัสดุ หรือ DONE เพื่อสรุป: ').strip()
        if name.lower() == 'done': break
        if not name: continue
        spec = input('สเปก/ขนาด: ').strip() or None
        qty = input('จำนวน: ').strip()
        unit = input('หน่วย: ').strip()
        budget_unit = input('ราคา BOQ ต่อหน่วย (Enter ถ้าไม่มี): ').strip() or None
        result = add_boq_item(project_id, input_name=name, specification=spec,
            quantity=qty, unit=unit, budget_unit_price=budget_unit, room_id=room_id)
        manual_count += 1
        print('Item ID:', result['item_id'], '| สถานะ:', result['match_status'])
    print('รับรายการแล้ว:', manual_count)

items_now = list_boq_items(project_id)
display(pd.DataFrame(items_now))

## 5. ตรวจและยืนยันการจับคู่

การจับคู่แบบ exact code/name/alias และหน่วยตรงกันจึง auto-match ได้ ส่วน fuzzy match เป็นเพียงคำแนะนำ ต้องเลือกยืนยันเอง รายการที่ยังไม่มีวัสดุเชื่อมโยงจะไม่ถูกนำไปคิดราคาประหยัดรวม

In [ ]:
pending = [row for row in list_boq_items(project_id) if row['match_status'] in {'NEEDS_REVIEW','UNMATCHED'}]
for item in pending:
    print(f"\nItem {item['item_id']}: {item['input_name']} | {item['input_specification']} | {item['input_unit']}")
    with connection() as conn:
        cur = conn.cursor()
        candidates = find_candidates(cur, item['input_name'], item['input_specification'], item['input_unit'])
    if not candidates:
        print('ยังไม่มี candidate ที่หน่วย/สเปกเข้ากันได้')
        continue
    for idx, candidate in enumerate(candidates, start=1):
        print(idx, candidate['canonical_name'], candidate['standard_unit'],
              candidate['specification'], f"score={candidate['confidence_score']:.0%}")
    choice = input('พิมพ์เลข candidate เพื่อยืนยัน หรือ Enter เพื่อข้าม: ').strip()
    if choice.isdigit() and 1 <= int(choice) <= len(candidates):
        confirm_boq_match(item['item_id'], candidates[int(choice)-1]['material_id'], actor='notebook_user')
        print('บันทึกการยืนยันและหลักฐานการจับคู่แล้ว')

## 6. สาธิต CRUD ใน Transaction ที่ rollback

เซลล์นี้ใช้ BOQ item จริงที่ผู้ใช้กรอก/นำเข้าเป็นต้นแบบ สาธิต Create, Read, Update และ Delete (soft delete แบบเดียวกับ `main.py`) ภายใน transaction แล้ว rollback เพื่อไม่ทิ้งแถวทดลองหรือเปลี่ยนข้อมูลโครงการจริง แต่บันทึกเหตุการณ์การสาธิตลง audit file แบบ append

In [ ]:
source_items = list_boq_items(project_id)
if not source_items:
    print('เพิ่ม BOQ จริงอย่างน้อยหนึ่งรายการก่อนสาธิต CRUD')
else:
    source = source_items[0]
    from boq_platform.db import connect
    conn = connect(); cur = conn.cursor(dictionary=True)
    try:
        conn.start_transaction()
        cur.execute('''INSERT INTO boq_items (project_id,room_id,material_id,source_row_no,
            source_product_code,input_name,input_specification,quantity,input_unit,budget_unit_price,match_status)
            SELECT project_id,room_id,material_id,source_row_no,source_product_code,input_name,
            input_specification,quantity,input_unit,budget_unit_price,match_status FROM boq_items WHERE item_id=%s''',
            (source['item_id'],))
        demo_item_id = cur.lastrowid
        cur.execute('SELECT item_id,input_name,quantity,input_unit FROM boq_items WHERE item_id=%s', (demo_item_id,))
        created_row = cur.fetchone(); print('CREATE + READ:', created_row)
        cur.execute('UPDATE boq_items SET quantity=quantity+1 WHERE item_id=%s', (demo_item_id,))
        cur.execute('SELECT item_id,quantity FROM boq_items WHERE item_id=%s', (demo_item_id,))
        print('UPDATE:', cur.fetchone())
        # Same soft delete as main.py: the row stays for audit, but is hidden from analytics.
        cur.execute('UPDATE boq_items SET is_deleted=1 WHERE item_id=%s', (demo_item_id,))
        cur.execute('SELECT COUNT(*) AS n FROM boq_items WHERE item_id=%s AND is_deleted=0', (demo_item_id,))
        print('DELETE (soft) verification — active rows:', cur.fetchone())
        conn.rollback()
        from boq_platform.audit import append_event
        append_event('crud_demo_rolled_back', project_id=project_id, source_item_id=source['item_id'],
                     operations=['CREATE','READ','UPDATE','DELETE'], rollback=True)
        print('ROLLBACK: ฐานข้อมูลกลับสู่สถานะก่อนสาธิต')
    finally:
        cur.close(); conn.close()

## 7. Insight, Top 3 และ Action Plan

SQL เป็นผู้คำนวณต้นทุน อันดับ และ coverage; Pandas ใช้จัดรูปผลลัพธ์เพื่อแสดงใน Notebook เท่านั้น Top 3 ร้านต่อวัสดุจะมีได้เท่าที่มีราคาสดและเทียบกันได้ ส่วนยอดรวมร้านจะจัดอันดับเฉพาะเมื่อครอบคลุม BOQ ครบ

In [ ]:
insight = generate_insights(project_id)
project = insight['project']
display(HTML(f"<h2>BOQ Dashboard: {project['project_name']}</h2>"
    f"<p>ขอบเขต: {project['scope_type']} | จังหวัด: {project['province_name'] or 'ไม่ระบุ'} | "
    f"รายการ {project['line_count']} | จับคู่ {project['matched_count']}</p>"
    f"<h3>ต้นทุน BOQ ที่มีราคา: ฿{(project['boq_cost'] or 0):,.2f}</h3>"))
official = insight['official_summary']
display(HTML('<h3>เทียบราคา BOQ กับราคากลาง สนค. (Dataset 2)</h3>'))
if official:
    print(f"เทียบได้ {official['lines_compared']}/{official['lines_total']} รายการ | BOQ ฿{official['boq_cost']:,.2f} "
          f"vs ราคากลาง ฿{official['official_cost']:,.2f} | ส่วนต่าง ฿{official['diff']:,.2f}")
    display(pd.DataFrame(insight['official_reference']))
else:
    print('ยังไม่มีรายการที่มีทั้งราคา BOQ และราคากลางในจังหวัด/หน่วยเดียวกัน')
display(pd.DataFrame(insight['top_cost_items']))
display(pd.DataFrame(insight['top_retail_offers']))
display(pd.DataFrame(insight['top_3_retailers']))
display(pd.DataFrame(insight['retailer_baskets']))
if insight['best_full_basket']:
    print('ร้านที่ครอบคลุม BOQ ครบ:', insight['best_full_basket']['source_name'])
    print('ต้นทุนตะกร้า:', insight['best_full_basket']['covered_total'])
    print('โอกาสประหยัดโดยประมาณ:', insight['estimated_savings'],
          '| percent:', insight['estimated_savings_pct'])
else:
    print('ยังไม่มีร้านที่มีราคาเทียบได้ครบทุกบรรทัด จึงไม่คำนวณยอดประหยัดรวม')

display(HTML('<h3>Action Plan</h3>'))
for action in insight['action_plan']:
    display(HTML(f"<b>[{action['priority']}] {action['keyword']}</b> — {action['evidence']}<br>"
        f"Next action: {action['next_action']}<br><br>"))

In [ ]:
# Thai-capable fonts (Windows: Tahoma / Leelawadee UI; macOS: Thonburi); falls back silently.
plt.rcParams['font.family'] = ['Tahoma', 'Leelawadee UI', 'Thonburi', 'Noto Sans Thai', 'DejaVu Sans']
cost_df = pd.DataFrame(insight['top_cost_items'])
if not cost_df.empty and 'boq_line_cost' in cost_df.columns:
    cost_df['boq_line_cost'] = pd.to_numeric(cost_df['boq_line_cost'])
    ax = cost_df.sort_values('boq_line_cost').plot.barh(x='material_name', y='boq_line_cost',
        legend=False, figsize=(9,4), color='#2676a8', title='Top BOQ Material Cost')
    ax.set_xlabel('BOQ line cost (THB)'); ax.set_ylabel('')
    plt.tight_layout(); plt.show()
else:
    print('ไม่มีราคา BOQ สำหรับสร้างกราฟต้นทุน')

basket_df = pd.DataFrame(insight['retailer_baskets'])
if not basket_df.empty:
    display(basket_df[['source_name','covered_lines','eligible_lines','coverage_ratio','covered_total','full_coverage']])
else:
    print('ยังไม่มีรายการที่จับคู่กับวัสดุและราคาได้')

## 8. ส่งออกผลและตรวจ Audit Log

ราคาและคำแนะนำที่ส่งออกเป็นผล ณ วันที่รัน Notebook ต้องตรวจยืนยันราคากับร้านก่อนตัดสินใจจัดซื้อ

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_path = EXPORT_DIR / f'boq_insight_project_{project_id}.xlsx'
with pd.ExcelWriter(export_path, engine='openpyxl') as writer:
    pd.DataFrame(insight['top_cost_items']).to_excel(writer, sheet_name='Top Costs', index=False)
    pd.DataFrame(insight['official_reference']).to_excel(writer, sheet_name='vs Official Price', index=False)
    pd.DataFrame(insight['top_retail_offers']).to_excel(writer, sheet_name='Retail Top 3', index=False)
    pd.DataFrame(insight['top_3_retailers']).to_excel(writer, sheet_name='Top 3 Stores', index=False)
    pd.DataFrame(insight['retailer_baskets']).to_excel(writer, sheet_name='Store Coverage', index=False)
    pd.DataFrame(insight['action_plan']).to_excel(writer, sheet_name='Action Plan', index=False)
print('Exported:', export_path)
for log_path in [LOG_DIR/'audit.jsonl', LOG_DIR/'match_audit.jsonl']:
    if log_path.exists():
        print(f'\n{log_path.name} — 5 เหตุการณ์ล่าสุด')
        for line in log_path.read_text(encoding='utf-8').splitlines()[-5:]: print(line)